# 04 — Métricas de Distância: Cosine, Dot Product e Euclidiana

## Por que a métrica importa?

Você tem dois embeddings e quer saber o quão parecidos são os textos. Como calcular essa "parecença"?
A fórmula matemática que você escolhe — a **métrica de distância** — impacta diretamente a qualidade dos resultados.

As três métricas mais usadas em busca vetorial:

| Métrica | O que mede | Range |
|---------|-----------|-------|
| **Cosine** | Ângulo entre os vetores | -1 a 1 (1 = idêntico) |
| **Dot Product** | Projeção de um vetor no outro | -∞ a +∞ |
| **Euclidiana (L2)** | Distância em linha reta | 0 a +∞ (0 = idêntico) |

**Spoiler:** para RAG com texto, a resposta quase sempre é Cosine (ou Dot Product com vetores normalizados, que é equivalente). Este notebook explica por quê — e os casos onde a escolha muda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Modelo carregado:", model.get_sentence_embedding_dimension(), "dimensões")

## 4.1 Similaridade de Cosseno

A similaridade de cosseno mede o **ângulo** entre dois vetores — ignora completamente o comprimento (norma).

**Fórmula:** `cos(θ) = (A · B) / (|A| × |B|)`

**Intuição geométrica:** dois vetores saindo da origem. Se apontam para a mesma direção (θ = 0°), cos = 1.
Se perpendiculares (θ = 90°), cos = 0. Se opostos (θ = 180°), cos = -1.

**Por que ignorar o comprimento?** Frases mais longas tendem a ter vetores com norma maior — não porque são
"mais importantes", mas porque têm mais tokens. Se usarmos distância euclidiana, frases longas pareceriam
sempre "mais distantes", mesmo sendo semanticamente idênticas a frases curtas. O cosseno elimina esse viés.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def dot_product(a, b):
    return np.dot(a, b)

def euclidean_distance(a, b):
    return np.linalg.norm(a - b)

# Frases de teste
frases = [
    "O gato dorme no sofá",           # 0
    "O felino repousa no divã",        # 1 — similar a 0
    "Machine learning usa gradiente",  # 2 — diferente
]
embs = model.encode(frases, normalize_embeddings=True)

print("Similaridade COSINE:")
for i in range(len(frases)):
    for j in range(i+1, len(frases)):
        sim = cosine_similarity(embs[i], embs[j])
        print(f"  '{frases[i][:30]}' × '{frases[j][:30]}' = {sim:.3f}")

### O que o cosseno nos diz?

- **~0.58** entre "gato dorme" e "felino repousa": alta similaridade, sem nenhuma palavra em comum. O modelo entendeu que ambas descrevem a mesma situação.
- **~0.13** entre texto de animal e texto de ML: pouca relação semântica.

Esse poder de capturar semântica sem sobreposição de palavras é o que torna embeddings + cosseno tão úteis para RAG.

## 4.2 Dot Product (Produto Escalar)

**Fórmula:** `A · B = Σ(aᵢ × bᵢ)`

**Relação com cosseno:** quando os vetores estão **normalizados** (norma = 1), dot product e cosine são **matematicamente idênticos**.

A maioria dos modelos modernos (incluindo all-MiniLM-L6-v2 com `normalize_embeddings=True`) retorna vetores normalizados. Nesses casos, você pode usar qualquer um — o resultado é o mesmo.

**Quando divergem:** se os vetores NÃO estão normalizados. Nesse caso, o dot product favorece vetores com norma maior (frases mais longas), o que geralmente é indesejado para texto.

**Vantagem computacional:** o dot product é ligeiramente mais rápido (sem a divisão pelas normas). Para bilhões de operações, isso importa.

In [ ]:
# Demonstração: com vetores normalizados, cosine == dot product
print("Com vetores NORMALIZADOS (norma=1):")
print(f"  Cosine = {cosine_similarity(embs[0], embs[1]):.6f}")
print(f"  Dot    = {dot_product(embs[0], embs[1]):.6f}")
print(f"  Diferença: {abs(cosine_similarity(embs[0], embs[1]) - dot_product(embs[0], embs[1])):.8f}")

# Sem normalização — agora divergem
embs_raw = model.encode(frases, normalize_embeddings=False)
print("
Com vetores NÃO normalizados:")
print(f"  Cosine = {cosine_similarity(embs_raw[0], embs_raw[1]):.6f}")
print(f"  Dot    = {dot_product(embs_raw[0], embs_raw[1]):.6f}")
print(f"  Diferença: {abs(cosine_similarity(embs_raw[0], embs_raw[1]) - dot_product(embs_raw[0], embs_raw[1])):.4f}")

## 4.3 Distância Euclidiana (L2)

**Fórmula:** `d(A, B) = √(Σ(aᵢ - bᵢ)²)`

É a "distância em linha reta" entre dois pontos no espaço vetorial.

**Quando faz sentido:** quando a *posição absoluta* no espaço importa — coordenadas GPS, pixels de imagem, dados físicos.

**Por que geralmente NÃO é ideal para texto:**
1. Afetada pela norma: frases mais longas têm vetores maiores, logo distâncias maiores
2. Em alta dimensão, as distâncias euclidianas ficam todas parecidas (maldição da dimensionalidade)
3. Não invariante a escala

**Exceção:** embeddings de imagem treinados com triplet loss (FaceNet, por exemplo) são otimizados para distância euclidiana.

In [ ]:
# Visualização 2D: por que cosine vs euclidean faz diferença

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Vetores 2D para ilustração
A = np.array([2.0, 1.0])   # vetor curto
B = np.array([4.0, 2.0])   # vetor longo, mesma direção de A
C = np.array([1.0, 2.0])   # vetor diferente

for ax, title, normalize in [(axes[0], "Vetores ORIGINAIS", False), (axes[1], "Vetores NORMALIZADOS", True)]:
    vecs = {"A": A.copy(), "B": B.copy(), "C": C.copy()}
    if normalize:
        vecs = {k: v/np.linalg.norm(v) for k, v in vecs.items()}

    colors = {"A": "blue", "B": "red", "C": "green"}
    for name, v in vecs.items():
        ax.annotate("", xy=v, xytext=(0,0), arrowprops=dict(arrowstyle="->", color=colors[name], lw=2))
        ax.text(v[0]+0.05, v[1]+0.05, f"{name} {tuple(v.round(2))}", color=colors[name], fontsize=10)

    # Mostrar métricas
    a, b, c_ = vecs["A"], vecs["B"], vecs["C"]
    ax.set_title(f"{title}
Cos(A,B)={cosine_similarity(a,b):.2f} | Euc(A,B)={euclidean_distance(a,b):.2f}
"
                 f"Cos(A,C)={cosine_similarity(a,c_):.2f} | Euc(A,C)={euclidean_distance(a,c_):.2f}")
    ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 3)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.grid(True, alpha=0.3)

plt.suptitle("A e B têm a mesma direção (só diferem em escala).
Cosine vê isso; Euclidiana não.", fontsize=12)
plt.tight_layout()
plt.show()

### Conclusão da visualização

No gráfico da esquerda (vetores originais):
- **A** e **B** apontam para a mesma direção — semanticamente idênticos
- Cosine(A, B) = 1.0 ✓ — captura que são iguais
- Euclidean(A, B) = grande ✗ — "acha" que são distantes por causa da escala

Isso ilustra por que **cosseno é a escolha correta para texto**: ele é invariante à norma do vetor. Documentos curtos e longos sobre o mesmo assunto ficam igualmente próximos.

## 4.4 Configurando Métricas no Qdrant

No Qdrant, a métrica de distância é configurada na criação da coleção — não pode ser mudada depois sem reindexar.

In [ ]:
try:
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams
    client = QdrantClient(host="localhost", port=6333)
    client.get_collections()
    QDRANT_OK = True
    print("Qdrant conectado")
except Exception as e:
    QDRANT_OK = False
    print(f"Qdrant offline ({e}) — mostrando só o código de configuração")

if QDRANT_OK:
    for metric_name, metric in [("COSINE", Distance.COSINE), ("DOT", Distance.DOT), ("EUCLID", Distance.EUCLID)]:
        col = f"demo_{metric_name.lower()}"
        if client.collection_exists(col):
            client.delete_collection(col)
        client.create_collection(col, vectors_config=VectorParams(size=384, distance=metric))
        print(f"Coleção '{col}' criada com {metric_name}")

## Resumo: Qual métrica escolher?

```
Tipo de dado?
├── Texto semântico (maioria dos casos RAG)
│   └── Vetores normalizados? → COSINE ou DOT (equivalentes)
│       Vetores não normalizados? → COSINE (mais seguro)
├── Imagens (FaceNet, ArcFace)
│   └── EUCLID (modelos treinados para isso)
└── Ranking com magnitude (DPR, ColBERT)
    └── DOT não normalizado (magnitude = relevância)
```

| Métrica | Fórmula | Quando usar |
|---------|---------|-------------|
| **Cosine** | cos(θ) = A·B / (\|A\|\|B\|) | **Padrão para RAG com texto** |
| **Dot Product** | A · B = Σaᵢbᵢ | Texto normalizado (= cosine) ou ranking com magnitude |
| **Euclidiana** | √Σ(aᵢ-bᵢ)² | Imagens, dados físicos, quando posição absoluta importa |

**Para o seu sistema RAG:** use `Distance.COSINE` no Qdrant. Você estará correto em 95%+ dos casos.

**Próximos passos:**
- [05 — Comparação de Modelos](05_models_comparison.html): qual modelo de embedding escolher?
- [01 — Qdrant Intro](../02_vector_databases/01_qdrant_intro.html): como criar coleções com cada métrica